# 19 · Final test v5 (session rules for real traffic)

The v4 final test (notebook 18) showed one real weakness: on a real shop log (zanbil) the session layer flagged 29.6% of
requests, because product pictures counted as ID enumeration and search-engine crawlers' dead links counted as scanning.
v5 changes only the session rules (ml/sessions.py): picture/script/style files and paging/cache-busting parameters are not
object IDs, a scan needs its not-found answers to be at least 10% of the session, pages are compared by shape, and every
swept resource is marked. The v4 result stays in the report as it is; this notebook tests v5 once, on data nobody has seen:

- **bak61** (both sites): a backup run made with generator v4 together with run6 and sealed unopened. It checks that the
  changed rules still catch the attacks.
- **zanbil test slice**: lines 7,000,001-7,200,000 of the zanbil log, cut before any v5 change and sealed unopened. It has
  no labels, so it measures false alarms on real traffic. The v5 rules were developed on two other slices
  (lines 1-200,000 and 3,000,001-3,200,000).

**Rules:** code frozen at tag `v5.0-final-test` (printed below, must say `(clean)`); every file's SHA-256 is checked first;
`RUN_FINAL = True` is run once; nothing is changed after seeing the result.

In [ ]:
RUN_FINAL = False   # set to True ONCE, after the code is frozen

# Filled in before the final run, never after:
ZANBIL_SHA256 = '92FB7572FD9E2B4C8353C55B09CEE240432187D46614EA367EBCF5DF89157087'  # test slice, hashed when cut (2026-11-02)
ZANBIL_LINES = 200_000  # the test slice is exactly 200,000 lines

In [ ]:
import hashlib, re, subprocess, sys, tempfile
from itertools import islice
from pathlib import Path, PureWindowsPath
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from backend.app.log_parser import decode, parse_log
from eval.final_test import (defence_summary, investigate, next_step_summary, recall_by_type, run, run_tool,
                             stage_summary, summary, type_accuracy, unusual_summary)

commit = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=ROOT, capture_output=True, text=True).stdout.strip()
tag = subprocess.run(['git', 'describe', '--tags', '--exact-match'], cwd=ROOT, capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(['git', 'status', '--porcelain', '--', 'ml', 'eval', 'backend'], cwd=ROOT,
                       capture_output=True, text=True).stdout.strip()
print('code commit:', commit, tag or '(no tag)', '(uncommitted changes in ml/eval/backend!)' if dirty else '(clean)')

D = ROOT / 'data'
# name: (log, request labels, session labels)
if RUN_FINAL:
    RUNS = {'Juice Shop (bak61)': (D / 'raw' / 'juiceshop_generated_bak61.log', D / 'labels' / 'requests_bak61.csv',
                                  D / 'labels' / 'sessions_bak61.csv'),
            'DVWA (bak61)': (D / 'raw' / 'dvwa_generated_bak61.log', D / 'labels' / 'requests_dvwa_bak61.csv',
                            D / 'labels' / 'sessions_dvwa_bak61.csv')}
    ZANBIL = D / 'raw' / 'zanbil_test_v5.log'
else:
    RUNS = {'Juice Shop (v4dev)': (D / 'raw' / 'juiceshop_generated_v4dev.log', D / 'labels' / 'requests_v4dev.csv',
                                   D / 'labels' / 'sessions_v4dev.csv'),
            'DVWA (v4dev)': (D / 'raw' / 'dvwa_generated_v4dev.log', D / 'labels' / 'requests_dvwa_v4dev.csv',
                             D / 'labels' / 'sessions_dvwa_v4dev.csv')}
    ZANBIL = D / 'raw' / 'public_apache_logs.log'   # stand-in: another public log, only to test the code

# Fingerprint of every file read. In the final run it must match the hash lists written when the files were made.
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest().upper()   # upper: same as Get-FileHash

def read_hashes(path):
    raw = Path(path).read_bytes()   # missing: ask for it, never skip the check
    text = raw.decode('utf-16') if raw[:2] in (b'\xff\xfe', b'\xfe\xff') else raw.decode('utf-8-sig')
    return {PureWindowsPath(p.strip()).name: h.upper() for h, p in re.findall(r'\b([0-9A-Fa-f]{64})\s+(.+)', text)}

files = [f for fs in RUNS.values() for f in fs] + ([ZANBIL] if ZANBIL.exists() else [])
hashes = {f.name: sha256(f) for f in files}
for file_name, h in hashes.items():
    print(h, file_name)
if RUN_FINAL:
    expected = {ZANBIL.name: ZANBIL_SHA256.upper()}
    for hash_list in ['bak61_hashes.txt']:
        expected.update(read_hashes(D / 'labels' / hash_list))
    changed = [f for f, h in hashes.items() if expected.get(f) != h]
    if changed:
        raise RuntimeError(f'Not as in the hash lists (changed, or missing from the lists): {changed}. '
                           'Stop here, do not look at any result.')
    print('all files match the hash lists')

results = {}
for name, (log, labels, _) in RUNS.items():
    req, sessions, report = run(log, labels)
    req, sessions = investigate(req, sessions)
    results[name] = (req, sessions)
    print(name, report)

## 1. Requests: how many attacks are caught, and how many false alarms

`rules` is the baseline: fixed attack patterns, no machine learning. `+ sessions` adds the session layer
(brute force, enumeration, scanning). A false alarm "on normal users" is a normal request in a normal user's session;
the others are ordinary pages an attacker loads between two attacks (inside a session that really is an attacker's).

In [ ]:
pd.concat({name: summary(req) for name, (req, _) in results.items()})

## 2. Recall by attack type

The newer families (ssrf, xxe, ssti, nosql, log4shell, ldap, xpath) are the ones the v4 generator added.

In [ ]:
pd.concat({name: recall_by_type(req) for name, (req, _) in results.items()})

## 3. Sessions: is every attacker found, and no normal user flagged?

`critical` = high risk and the attacker reached exploitation or data theft.

In [ ]:
pd.concat({name: pd.crosstab(s['truth'], s['risk_level']) for name, (_, s) in results.items()})

## 4. Attack type of the caught payload attacks

In [ ]:
for name, (req, _) in results.items():
    table, acc = type_accuracy(req)
    print(f'{name}: {acc:.1%} correct')
    display(table)

## 5. Stage, next step and defence

**Stage**: the furthest stage each session reached. Normal users should have none.

**Next step**: only the campaign sessions (attackers that do several kinds of attack one after the other). *Order read exactly* = the steps read from the log are the steps the generator ran. Top-1 / top-3 = how often the real next step is the model's first guess / among its first three, on the true order and on the order read from the log.

**Defence**: the right action comes from the labels: any payload attack -> block, only brute force, enumeration or scanning -> rate limit, normal -> allow. The two mistakes that matter are an attacker allowed and a normal user blocked.

In [ ]:
pd.concat({name: stage_summary(s) for name, (_, s) in results.items()})

In [ ]:
pd.DataFrame({name: next_step_summary(req, RUNS[name][2]) for name, (req, _) in results.items()})

In [ ]:
pd.DataFrame({name: defence_summary(req, s) for name, (req, s) in results.items()})

## 6. The unusual-request mark

Review only: it never changes a label or a risk. It marks the most unusual requests of each log (Isolation Forest fitted on that log). The rows that matter: how many normal requests it marks, and whether it marks any attack the detector and the session layer missed.

In [ ]:
pd.DataFrame({name: unusual_summary(req) for name, (req, _) in results.items()})

## 7. A real public log (zanbil test slice): false alarms on real traffic

The log has no labels, so every request is treated as normal and every flag is listed for review. Only the first
`ZANBIL_LINES` lines are used, so the run fits on a laptop. A flag is not automatically wrong: a real shop log also
holds real scanners and bots. Section 9 writes down, for each kind of flag, whether it is a real probe or a mistake.

In [ ]:
if ZANBIL.exists():
    sample = Path(tempfile.mkdtemp()) / 'public_sample.log'
    with open(ZANBIL, 'rb') as src, open(sample, 'wb') as dst:
        dst.writelines(islice(src, ZANBIL_LINES))
    zreq, zses, zrep = run_tool(sample, attacker_ip='none')   # no labels: everything counted normal
    zreq, zses = investigate(zreq, zses)
    print(zrep)
    print('requests:', len(zreq), '| IPs:', zreq['ip'].nunique(), '| sessions:', len(zses))
    flags = pd.Series({'flagged by model': int(zreq['pred'].sum()),
                       'flagged by sessions only': int(((zreq['pred'] == 0) & (zreq['session_attack'] == 1)).sum()),
                       'flagged (model + sessions)': int(zreq['model + sessions'].sum()),
                       'flagged by rules': int(zreq['rules'].sum()),
                       'share of requests flagged (model + sessions)': round(float(zreq['model + sessions'].mean()), 4),
                       'marked unusual': int(zreq['unusual'].sum())})
    display(flags)
    display(pd.crosstab(zses['risk_level'], zses['action']))
else:
    print('public log not on this machine: section skipped')

In [ ]:
if ZANBIL.exists():
    pd.set_option('display.max_colwidth', 120)
    print('flagged sessions (medium and above):')
    display(zses[zses['risk_level'].isin(['medium', 'high', 'critical'])]
            [['ip', 'requests', 'attack_requests', 'risk_level', 'action', 'stage', 'reasons']].head(50))
    print('requests flagged by the model:')
    display(zreq.loc[zreq['pred'] == 1, ['ip', 'method', 'path', 'status', 'type_pred']].head(100))
    print('session-only flags, by behaviour:')
    display(zreq.loc[(zreq['pred'] == 0) & (zreq['session_attack'] == 1), 'behaviour'].value_counts())

## 8. Same logs without request bodies

Most real access logs do not store the body. This shows what is lost (attacks sent in a POST body) and checks the false alarms.

In [ ]:
pd.concat({name: summary(run(log, labels, drop_body=True)[0]) for name, (log, labels, _) in RUNS.items()})

## 9. What this shows

*(Write this after the one final run, from the numbers above. Do not change the code or rerun.)*